<div align="center">
  <img src="https://raw.githubusercontent.com/cvail-research/Hands-on-Computer-Vision/main/sesiones/sesion11/assets/NLP%20SESION/BANNER_SESIONES_SEMILLERO.png">
</div>

# <font color='#fabf61'>**Hands-On Sesión 11: Natural Language Processing 🔤🤖💬**</font>

## <font>**Tabla de Contenidos**</font>

**1. LLMs 🤖**
  - 1.1 Preprocesamiento del texto y tokenización (*naive* → tokens especiales → BPE)
  - 1.2 Arquitectura GPT-2: atención, predicción del siguiente token y generación

**2. Algunas tareas de NLP 💻🥸**
  - 2.1 Named Entity Recognition (spaCy)
  - 2.2 Text to Speech (Silero)

## <font color='#FC9797'>**0. Setup**</font>

In [ ]:
#@title Instalación de librerías
!pip install -q bertviz
!pip install -q -U transformers
!python -m spacy download es_core_news_lg  # modelos disponibles: sm | md | lg


In [ ]:
#@title Descargar los recursos de la sesión
# Los 4 archivos viven en el repo del semillero, dentro de sesiones/sesion11/resources.
# Esa carpeta fue BORRADA de la rama principal, por eso la URL fija el commit 15b4b6b
# (el mismo commit al que apunta el banner de arriba).
import os, sys, urllib.request

BASE = ("https://raw.githubusercontent.com/semilleroCV/Hands-on-Computer-Vision/"
        "15b4b6b851f9044910f7912d9d3d0282421a3f11/sesiones/sesion11/resources")

RES = "/content"                    # en Colab ya existe
os.makedirs(RES, exist_ok=True)

for fname in ["visualization.py",   # módulo de Python: define TokenVisualization
              "betty.txt",          # corpus para el tokenizer
              "split1.mx-news.txt", # dataset CoNLL para NER
              "tokenizer.png"]:     # diagrama del pipeline
    dest = os.path.join(RES, fname)
    if os.path.exists(dest):
        print("ya existe:  ", fname)
        continue
    urllib.request.urlretrieve(f"{BASE}/{fname}", dest)
    print("descargado: ", fname, f"({os.path.getsize(dest):,} bytes)")

# visualization.py se importa como módulo, así que su carpeta debe estar en sys.path
if RES not in sys.path:
    sys.path.insert(0, RES)


In [ ]:
#@title Imports unificados

# utils
import os, re
from collections import Counter
import pandas as pd
import numpy as np

# nlp tradicional
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import spacy
from spacy.tokens import Span
from spacy import displacy

# visualización
from IPython.display import display, HTML, Image as IPyImage, Audio
from tabulate import tabulate
from graphviz import Digraph
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Output, VBox, Layout

from visualization import TokenVisualization          # helper de la sesión
from bertviz.neuron_view import show
from bertviz import head_view
from bertviz.transformers_neuron_view import GPT2Model, GPT2Tokenizer

# torch / transformers
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForCausalLM, logging,
)

from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

logging.set_verbosity_error()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## <font color='#FC9797'>**1. LLMs 🤖**</font>
Antes de la llegada de los LLMs, los métodos tradicionales destacaban en tareas de categorización como la clasificación de correo no deseado y el reconocimiento directo de patrones, que podían capturarse con reglas personalizadas o modelos más simples. Sin embargo, solían tener un rendimiento inferior en tareas lingüísticas que exigían capacidades complejas de comprensión y generación, como el análisis sintáctico de instrucciones detalladas, el análisis contextual y la creación de textos originales coherentes y contextualizados.

El éxito de los LLM se puede atribuir a la arquitectura del **transformer** en la que se apoyan muchos LLMs, y a la gran cantidad de datos con los que se entrenan, lo que les permite capturar una amplia variedad de matices, contextos y patrones lingüísticos que serían difíciles de codificar manualmente.

### <font color='#EC91CE'>**1.1 Tokenizar y preprocesar texto**</font>
La tokenización es el proceso de convertir texto en una secuencia de tokens, que pueden ser palabras, subpalabras o caracteres. Estos tokens son las unidades de significado más pequeñas de un texto que un modelo de lenguaje puede procesar. Este proceso simplifica el texto y ayuda al modelo a trabajar con bloques de datos más manejables.

In [ ]:
#@title Pipeline de tokenización + carga del corpus
display(IPyImage('/content/tokenizer.png', width=700))

with open("/content/betty.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total de caracteres:", len(raw_text))
print("300 primeros caracteres:", raw_text[None])

#### Naive approach
Dividir cada palabra y carácter especial en tokens distintos y asignarles un token ID. Para ello construimos un vocabulario, el cual definirá cómo mapearemos cada palabra y carácter único.

In [ ]:
#@title Limpieza + construcción del vocabulario
def limpiar_texto(texto):
    texto = texto.lower()                                        # pasar a minúsculas
    texto = re.sub(r'[\n\r\t]', ' ', texto)                     # eliminar saltos de línea, tabs, etc.
    texto = ''.join(c for c in texto if c.isprintable())         # eliminar caracteres no imprimibles
    tokens = re.findall(r'\w+|[¿¡,.:;?_!"()\[\]\'\-\—]', texto)  # tokenizar por palabra y signo de puntuación
    return tokens

preprocessed = limpiar_texto(raw_text)
print('100 primeros tokens:', preprocessed[:99])
print('Cantidad de tokens:', len(preprocessed))

all_words = sorted(None(preprocessed))
vocab_size = len(None)
print('Tamaño de nuestro vocabulario:', vocab_size)

vocab = {token: integer for integer, token in enumerate(all_words)}
print('20 primeros tokens únicos junto a sus respectivos IDs:')
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

In [ ]:
#@title Clase para codificar texto en tokens y decodificar tokens en texto
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = limpiar_texto(text)
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'\-])', r'\1', text)
        return text

In [ ]:
#@title Visualizar tokens de un fragmento
tokenizer = SimpleTokenizerV1(vocab)

# puedes cambiar múltiples parámetros para modificar el visualizer!
token_viz = TokenVisualization(
    encoder=tokenizer.encode,
    decoder=tokenizer.decode,
    font_family='Arial',
)

tokens = tokenizer.encode(raw_text)               # codificar texto en token IDs
fragmento_tokens = None[600:670]                # escoger un fragmento de estos tokens
sample_text = tokenizer.decode(fragmento_tokens)  # decodificar estos tokens en texto

HTML(token_viz.visualize(sample_text))

In [ ]:
#@title RETO: ¿qué ocurre si pasamos palabras no presentes en el texto original?
texto = ''  # @param {type:"string"}
HTML(token_viz.visualize(texto))

**↑↑↑↑ ¿Por qué ocurre esto?** El vocabulario es cerrado: cualquier palabra fuera de él no tiene ID. La solución mínima son los **tokens especiales**.

In [ ]:
#@title Vocabulario extendido + tokenizer con tokens especiales
all_tokens = sorted(list(set(preprocessed)))
all_tokens.None(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print('Tamaño del vocabulario:', len(vocab))   # el tamaño de nuestro vocabulario subió!
print('Últimos 5 tokens')
for item in list(vocab.items())[-5:]:
    print(item)


def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[\n\r\t]', ' ', texto)
    texto = ''.join(c for c in texto if c.isprintable())
    # primero identifica tokens como <|endoftext|>, luego palabras y signos
    tokens = re.findall(r'<\|.*?\|>|\w+|[¿¡,.:;?_!"()\[\]\'\-\—]', texto)
    return tokens


class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
        self.unk_token = "<|unk|>"   # acá!

    def encode(self, text):
        preprocessed = limpiar_texto(text)
        # reemplaza palabras desconocidas con <|unk|>
        preprocessed = [t if t in self.str_to_int else self.None for t in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([¿¡,.?!"()\'\-])', r'\1', text)
        return text

In [ ]:
#@title Probar el tokenizer V2 con dos textos separados por <|endoftext|>
tokenizer = SimpleTokenizerV2(vocab)
token_viz = TokenVisualization(
    encoder=tokenizer.encode,
    decoder=tokenizer.decode,
    font_family='Arial',
    # cmap='tab10'
)

texto1 = 'hola'  # @param {type:"string"}
texto2 = 'me llamo'  # @param {type:"string"}
texto = "<|endoftext|>".join((texto1, texto2))

HTML(token_viz.visualize(texto))

#### Byte Pair Encoding (BPE)
BPE descompone las palabras que no están en su vocabulario predefinido en subpalabras más pequeñas o incluso en caracteres individuales, lo que le permite manejar palabras fuera del vocabulario. Así, si el tokenizador encuentra una palabra desconocida durante la tokenización, puede representarla como una secuencia de subpalabras o caracteres. Se denomina *Byte* porque se hace a nivel de bytes.

**Pasos**
1. Obtener el vocabulario inicial de tokens
2. Tokenizar el texto con estos tokens base
3. Contar pares de tokens consecutivos
4. Seleccionar el par de tokens más frecuente y crear un nuevo token que sea la combinación del par más frecuente
5. Repetir desde el paso 3 hasta alcanzar el número deseado de combinaciones o vocabulario

In [ ]:
#@title ¿Por qué no un vocabulario de palabras completo? El costo en memoria
vocab_size = 93000    # número de palabras en la RAE
embedding_dim = 768   # dimensión de cada vector de embedding de GPT-2
bytes_per_float = 4   # 32 bits = 4 bytes por número flotante

total_bytes = vocab_size * embedding_dim * bytes_per_float
print("Memoria requerida:")
print(f"- Bytes: {total_bytes:,}")
print(f"- KB: {total_bytes/1024:,.2f}")
print(f"- MB: {total_bytes/1024**2:,.2f}")

In [ ]:
#@title Clase para manejar BPE
class BPE:
    def __init__(self):
        self.dictionary = {i: [i] for i in range(256)}

    def encode(self, text):
        return list(text.encode('utf-8'))  # convierte texto a lista de IDs de bytes

    def decode(self, ids):
        if isinstance(ids, int):
            if ids <= 255:
                # token base (byte), decodificar directamente
                return bytes([ids]).decode('utf-8', errors='replace')
            else:
                # token merge: buscar en diccionario y decodificar recursivamente
                if ids not in self.dictionary:
                    raise ValueError(f"Token {ids} no está en el diccionario")
                return self.decode(self.dictionary[ids])
        elif isinstance(ids, list):
            # si todos los elementos son bytes (ints entre 0 y 255), decodificarlos juntos
            if all(isinstance(i, int) and 0 <= i <= 255 for i in ids):
                return bytes(ids).decode('utf-8', errors='replace')
            else:
                # si hay tokens merge en la lista, decodificarlos recursivamente y concatenar
                return ''.join(self.decode(i) for i in ids)
        else:
            raise ValueError("Entrada inválida para decode")

    def merge(self, ids, pair, idx):
        newids = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
                newids.append(idx)
                i += 2
            else:
                newids.append(ids[i])
                i += 1
        # guarda en el diccionario la concatenación de los bytes de ambos tokens del merge
        self.dictionary[idx] = self.dictionary[pair[0]] + self.dictionary[pair[1]]
        return newids

    def decode_merges_dict(self, merges):
        # decodificar merges en caracteres
        decoded_dict = {}
        for pair, idx in merges.items():
            decoded_pair = tuple(self.decode(token) for token in pair)
            decoded_dict[decoded_pair] = self.decode(idx)
        return decoded_dict

In [ ]:
#@title Pasos 1-2: vocabulario base de bytes y tokenización
tokenizer = BPE()
sample_text = ('Pablito clavó un clavito en la calva de un calvito. '
               'Un clavito clavó Pablito en la calva de un calvito')

pre_proc = re.split(r'([¿¡,.:;?_!"()\'\-\—]|--|\s+)', sample_text)
pre_proc = [item.strip() for item in pre_proc if item.strip()]

tokens = tokenizer.encode(sample_text)

print('Cantidad de palabras/símbolos:', len(pre_proc))
print('Cantidad de bytes/tokens:', len(tokens))  # ¿por qué son más que el número de caracteres?

df = pd.DataFrame({
    'Byte ID': tokens,
    'Char': [bytes([b]).decode('utf-8', errors='replace') for b in tokens]
})  # este es nuestro vocabulario
display(df.head())

# frecuencia de palabras del texto original
freqs = Counter(pre_proc)
print(f"\n{'Palabra':<10} {'Frecuencia':>10}")
print("-" * 22)
for word, freq in freqs.most_common(20):
    print(f"{word:<10} {freq:>10}")

In [ ]:
#@title Paso 3: contar pares de tokens consecutivos
def get_stats(tokens):
    stats = {}
    for tok1, tok2 in zip(tokens, tokens[1:]):
        if (tok1, tok2) in stats:
            stats[(tok1, tok2)] += 1
        else:
            stats[(tok1, tok2)] = 1
    return stats

stats = get_stats(tokens)

print(f"{'Par':<10} {'Frecuencia':>10}")
print("-" * 22)
for i, (pair, freq) in enumerate(stats.items()):
    if i > 15:
        break
    print(f"{pair} {freq:>10}")

In [ ]:
#@title Pasos 4-5: hacer merge entre los pares más usados
vocab_size_final = 276
vocab_size_original = 256
num_merges = vocab_size_final - vocab_size_original
ids = list(tokens)  # copiamos para no destruir la lista original de tokens

merges = {}
for i in range(num_merges):
    stats = get_stats(tokens)
    top_pair = max(stats, key=stats.get)
    idx = vocab_size_original + i

    chars = tokenizer.decode(list(top_pair))
    print(f"merging {top_pair}('{chars}') into a new token {idx}")

    tokens = tokenizer.merge(tokens, top_pair, idx)
    merges[top_pair] = idx

In [ ]:
#@title Visualizar merges
def visualize_merges_graphviz(merges):
    dot = Digraph()
    dot.attr(fontname='Arial', rankdir='TB')   # vista de arriba a abajo (usa 'LR' para izq->der)
    dot.node_attr.update(fontname='Arial', fontsize='12')
    dot.edge_attr.update(fontname='Arial', fontsize='10')

    colors_a = ['#c9ffbf', '#be9bff', '#ff9bb6', '#c9ffbf']
    colors_b = ['#fff2d1', '#a0fff0', '#ff807e', '#bae6ff']
    colors_merged = ['#25bcc6', '#ffc3f2', '#5ec1ff', '#ffa582']

    for i, ((a, b), merged) in enumerate(merges.items()):
        ca, cb, cm = colors_a[i % 4], colors_b[i % 4], colors_merged[i % 4]
        dot.node(str(a), f"{a}", shape="circle", style="filled", fillcolor=ca, color=ca)
        dot.node(str(b), f"{b}", shape="circle", style="filled", fillcolor=cb, color=cb)
        dot.node(str(merged), f"{merged}", shape="box", style="filled,rounded", fillcolor=cm, color=cm)
        dot.edge(str(a), str(merged))
        dot.edge(str(b), str(merged))
    return dot

visualize_merges_graphviz(tokenizer.decode_merges_dict(merges))

### <font color='#EC91CE'>**1.2 GPT-2**</font>
Pasamos de un tokenizador artesanal a un modelo real. Primero inspeccionamos el mecanismo de **atención** (vectores *query* / *key*), luego cargamos un GPT-2 en español para ver tokenización BPE real, predicción del siguiente token y generación.

In [ ]:
#@title Visualizar vectores query/key para computar atención (bertviz)
text = "What is Ana's name?"
model_type = 'gpt2'
model_version = 'gpt2'
viz_model = GPT2Model.from_pretrained(model_version, output_attentions=None)
viz_tokenizer = GPT2Tokenizer.from_pretrained(model_version, do_lower_case=True)
show(viz_model, model_type, viz_tokenizer, text)

In [ ]:
#@title Cargar GPT-2 en español
tokenizer = AutoTokenizer.from_pretrained("datificate/gpt2-small-spanish")
model = AutoModelForCausalLM.from_pretrained(
    "datificate/gpt2-small-spanish",
    output_attentions=True
)
model_size = sum(t.numel() for t in model.parameters())
print(f"El tamaño del modelo es: {model_size/1000**2:.1f}M parámetros")
print('El context length es:', model.config.n_positions)

In [ ]:
#@title Tokenizer basado en BPE (real) + visualización
text = 'Hola, a mi me gusta el NLP'  # @param {type:"string"}
inputs = tokenizer(
    text,
    return_tensors='pt',
    return_offsets_mapping=True
)

token_viz = TokenVisualization(
    encoder=tokenizer.encode,
    decoder=tokenizer.decode,
    font_family='Arial',
)
HTML(token_viz.visualize(text))

In [ ]:
#@title Forward pass: top-5 predicciones del siguiente token
input_ids = inputs['input_ids']            # obtener los token ids
offsets = inputs["offset_mapping"]
tokens = [text[s:e] for s, e in offsets[0]]

# eliminar offset_mapping para pasarlo al modelo
model_inputs = {k: v for k, v in inputs.items() if k != "offset_mapping"}

with torch.no_grad():
    outputs = model(**model_inputs, output_hidden_states=True)

logits = outputs.logits                    # [1, seq_len, vocab_size]

for i, token in enumerate(tokens):
    probs = F.None(logits[0, i], dim=-1)
    topk = torch.topk(probs, 5)
    pred_tokens = [tokenizer.decode([idx]) for idx in topk.indices]
    print(f"\nToken '{token}' → Predicciones más probables:")
    for j, pred in enumerate(pred_tokens):
        print(f"  Top {j+1}: {repr(pred)}")

In [ ]:
#@title Visualizar las cabezas de atención
attention = outputs.attentions
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze())
head_view(attention, tokens)

In [ ]:
#@title Generar texto con nuestro modelo GPT
def generate_text(initial_text, model, tokenizer, max_new_tokens=100,
                  do_sample=True, temperature=1.0):
    with torch.no_grad():
        inputs = tokenizer(initial_text, return_tensors='pt')   # tokenizamos texto inicial
        outputs = model.generate(                               # generamos texto
            inputs['input_ids'],
            max_new_tokens=max_new_tokens,   # tokens nuevos a generar
            do_sample=do_sample,             # sampling aleatorio del nuevo token
            temperature=temperature,         # creatividad: <1 aburrido, >1 creativo
            pad_token_id=tokenizer.eos_token_id
        )
    # obtener secuencia generada y decodificar los ids a texto
    generated_ids = outputs
    return tokenizer.decode(generated_ids[0].tolist(), skip_special_tokens=True)


output = generate_text(text, model, tokenizer, max_new_tokens=20)
print("=== Texto completo ===")
display(HTML(f"<pre style='white-space: pre-wrap; word-wrap: break-word;'>{output}</pre>"))

In [ ]:
#@title ¿Qué sucede si le pasamos un texto largo al modelo? (context length)
output = generate_text(raw_text, model, tokenizer)
output

## <font color='#FC9797'>**2. Algunas tareas de NLP 💻🥸**</font>

### <font color='#EC91CE'>**2.1 Named Entity Recognition con spaCy**</font>
spaCy es una biblioteca de Python de código abierto especializada en NLP. Cuenta con varios modelos entrenados para analizar texto y predecir diferentes atributos lingüísticos basados en el contexto de cada palabra u oración. Una de estas tareas es:

**Reconocimiento de entidades nombradas (Named Entity Recognition - NER)**: detectar y clasificar en categorías específicas los nombres propios o conceptos importantes en el texto, como personas, lugares, fechas, organizaciones, cantidades, etc.

Existen varias formas de etiquetar datasets en NER, algunas de estas son:
- IO
- IOB/BIO
- IOBES/BIOES

Sus acrónimos se deben a las reglas:
 I → “inside”, O → “outside”, B → “beginning”, E → “end”, S → “single token entity”

Se ve tal que:
```
Patricio  B-PER
Perez  I-PER
asiste O
a  O
Hands-On B-ORG
```

In [ ]:
#@title Leer el dataset en formato CoNLL y extraer una oración
def read_conll_file(f):
    data = []
    with open(f) as i:
        sentences = i.read().strip().split("\n\n")
    for sentence in sentences:
        data.append([token.split() for token in sentence.split("\n")])
    return data


data = read_conll_file("/content/split1.mx-news.txt")
# ejemplo de las oraciones que hay en nuestro dataset
print(tabulate(data[49][1:], headers=['Sentence', '#', 'Word', 'POS', 'NER'], tablefmt='grid'))


def qm(data, sentence_id):
    """Obtener una oración del dataset como string junto a sus NER tags."""
    idx = int(sentence_id) - 1
    rows = data[idx]
    # filtrar filas que corresponden a la oración solicitada (se omite el encabezado)
    words = [row[2] for row in rows[1:] if str(row[1]) == str(sentence_id)]
    NER = [row[4] for row in rows[1:] if str(row[1]) == str(sentence_id)]
    return ' '.join(words), NER


sentence, ner = qm(data, str(50))
print('\nOración:', sentence)

In [ ]:
#@title Función para pasar de IOBES a spans de spaCy
def iobes_to_spacy_entities(doc, labels):
    entities = []
    start = None
    entity_label = None

    for i, tag in enumerate(labels):
        if tag == 'O':                                  # Outside
            if start is not None:
                entities.append(Span(doc, start, i, label=entity_label))
                start = None
                entity_label = None
        else:
            prefix, label = tag.split('-')
            if prefix == 'B':                           # Beginning
                if start is not None:
                    entities.append(Span(doc, start, i, label=entity_label))
                start = i
                entity_label = label
            elif prefix == 'I':                         # Inside
                pass
            elif prefix == 'E':                         # End
                if start is not None:
                    entities.append(Span(doc, start, i + 1, label=label))
                    start = None
                    entity_label = None
            elif prefix == 'S':                         # Single
                entities.append(Span(doc, i, i + 1, label=label))
                start = None
                entity_label = None

    # por si quedó una entidad abierta al final
    if start is not None:
        entities.append(Span(doc, start, len(labels), label=entity_label))

    doc.ents = entities
    return doc

In [ ]:
#@title Cargar el modelo de spaCy, inspeccionar entidades y tokenizar
nlp = spacy.load("es_core_news_lg")
print('Entidades que tiene este modelo:', nlp.pipe_labels['ner'])   # todas las entidades del modelo
print('La entidad PER representa:', spacy.None("PER"))           # .explain para entender una entidad

doc = nlp(sentence)
sentence_spans = list(doc.sents)
print('--- Oraciones:')
for i, sen in enumerate(sentence_spans):  # spacy divide el texto en oraciones y lo convierte en tokens
    print(f"[Oración {i}] {sen}")

print('--- Tokens:', [token for token in doc])

# los atributos ent_iob_ y ent_type_ permiten ver las entidades
headers = ['Palabra', 'IOB tag', 'Tipo de entidad']
entities = [(t.orth_, t.ent_iob_, t.ent_type_) for t in doc]
print(tabulate(entities[:10], headers=headers, tablefmt='grid'))

In [ ]:
#@title Entidades predichas vs. correctas + otras tareas (POS / dependencias)
print("Entidades PREDICHAS por spaCy:")
displacy.render(doc, style="ent", jupyter=True)

print("Entidades CORRECTAS (ground truth del dataset):")
doc = iobes_to_spacy_entities(doc, ner)
displacy.render(doc, style="ent", jupyter=True)

# spaCy también sirve para POS tagging y análisis de dependencias
displacy.render(sentence_spans[0], style="dep", jupyter=True)

### <font color='#EC91CE'>**2.2 Text to Speech 🔊**</font>
La tarea inversa: generar voz a partir de texto. Usamos los modelos Silero vía `torch.hub`.

In [ ]:
#@title TTS con Silero
language = 'en'
speaker = 'lj_16khz'
tts_model, symbols, sample_rate, _, apply_tts = torch.hub.load(
    repo_or_dir='snakers4/silero-models',
    model='silero_tts',
    language=language,
    speaker=speaker
)
tts_model = tts_model.to(device)

example_text = ('Hands On Computer Vision is a research group with the mission of training '
                'the next generation of computer vision experts through research and '
                'collaborative experiences.')

audio = apply_tts(texts=[example_text],
                  model=tts_model,
                  sample_rate=sample_rate,
                  symbols=symbols,
                  device=device)

display(Audio(audio[0], rate=sample_rate))